## Welcome to Week 6 Day 3: Context Engineering with MCP

People used to talk about Prompt Engineering, but now it's given way to a new Skill "Context Engineering".

Philipp Schmid of Google DeepMind wrote the seminal post about Context Engineering:

https://www.philschmid.de/context-engineering

For this week, we will put Context Engineering into practice, heavily using MCP servers.

1. Long-term memory: a knowledge graph the agent writes to and reads back
2. Web search: fresh information from the live web
3. Agentic RAG: a vector store the agent fills from its own research, then searches
4. Integrations: connecting to a live external service, with a local fallback

In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio, create_static_tool_filter
import os
from pathlib import Path
from datetime import datetime
from IPython.display import Markdown, display

load_dotenv(override=True)

True

### A quick note for Windows

Launching a local MCP server from a notebook hits one rough edge on Windows. The server writes its startup output to stderr, but inside a Windows Jupyter kernel that stream has no real file handle behind it, so the launch fails with an `io.UnsupportedOperation: fileno` error, while Mac and Linux are unaffected.

The fix is to send the server's stderr to the null device, so it always has somewhere real to write. We do that once in the next cell, which lets every cell below use `MCPServerStdio` exactly as the OpenAI Agents SDK documents it. On Mac and Linux it costs nothing beyond keeping the server's startup banner out of the notebook.

In [2]:
# On Windows, a stdio MCP server started from a Jupyter kernel writes to a stderr stream with no
# real file descriptor and crashes with io.UnsupportedOperation: fileno. We send the server's
# stderr to the null device so it always has somewhere real to write, which lets every cell below
# use MCPServerStdio exactly as the OpenAI Agents SDK documents it. Mac and Linux are unaffected.
import functools
import subprocess
import agents.mcp.server

agents.mcp.server.stdio_client = functools.partial(agents.mcp.server.stdio_client, errlog=subprocess.DEVNULL)

In [3]:
tavily_params = {"command": "npx", "args": ["-y", "tavily-mcp@latest"], "env": {"TAVILY_API_KEY": os.getenv("TAVILY_API_KEY")}}

async with MCPServerStdio(params=tavily_params, client_session_timeout_seconds=60) as server:
    tavily_tools = await server.list_tools()

tavily_tools

[Tool(name='tavily_search', title=None, description='Search the web for current information on any topic. Use for news, facts, or data beyond your knowledge cutoff. Returns snippets and source URLs.', inputSchema={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Search query'}, 'search_depth': {'type': 'string', 'enum': ['basic', 'advanced', 'fast', 'ultra-fast'], 'description': "The depth of the search. 'basic' for generic results, 'advanced' for more thorough search, 'fast' for optimized low latency with high relevance, 'ultra-fast' for prioritizing latency above all else", 'default': 'basic'}, 'topic': {'type': 'string', 'enum': ['general'], 'description': 'The category of the search. This will determine which of our agents will be used for the search', 'default': 'general'}, 'time_range': {'type': 'string', 'description': 'The time range back from the current date to include in the search results', 'enum': ['day', 'week', 'month', 'year']}, 'start_date':

In [4]:
search_only = create_static_tool_filter(allowed_tool_names=["tavily_search"])

Tavily's server offers several tools (search, extract, crawl, map, research). For this lab we only want plain web search, so we restrict the server to `tavily_search`. The OpenAI Agents SDK lets you hand an agent just the tools you choose with a static tool filter. Curating an agent's tools like this is itself context engineering.

## Part 3: Agentic RAG

RAG, retrieval augmented generation, means giving a model relevant documents to ground its answer. The usual setup loads documents into a vector store up front. Agentic RAG turns that around: the agent builds the knowledge base itself, deciding what is worth keeping and storing it as it works.

We use the official Qdrant MCP server. Qdrant is a vector database, and the server exposes two tools: one stores a piece of text, the other finds the most relevant stored text for a query. It runs fully locally here. `QDRANT_LOCAL_PATH` keeps everything on disk with no separate database to run, and it embeds text with a local model, so there is no extra API key. The first store or search downloads that small embedding model, so it pauses once on first use.

https://github.com/qdrant/mcp-server-qdrant

We hand one agent both Tavily and Qdrant: it researches a topic on the web, stores what it learns, then answers from its own knowledge base.

In [5]:
vectordb_path = Path("memory/qdrant")
vectorstore_params = {
    "command": "uvx",
    "args": ["mcp-server-qdrant"],
    "env": {
        "QDRANT_LOCAL_PATH": str(vectordb_path),
        "COLLECTION_NAME": "knowledge",
    },
}

async with MCPServerStdio(params=vectorstore_params, client_session_timeout_seconds=120) as server:
    vectorstore_tools = await server.list_tools()

vectorstore_tools

[Tool(name='qdrant-find', title=None, description='Look up memories in Qdrant. Use this tool when you need to: \n - Find memories by their content \n - Access memories for further analysis \n - Get some personal information about the user', inputSchema={'properties': {'query': {'description': 'What to search for', 'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'type': 'object'}, outputSchema=None, icons=None, annotations=None, meta=None, execution=None),
 Tool(name='qdrant-store', title=None, description='Keep the memory for later use, when you are asked to remember something.', inputSchema={'properties': {'information': {'description': 'Text to store', 'title': 'Information', 'type': 'string'}, 'metadata': {'anyOf': [{'additionalProperties': True, 'type': 'object'}, {'type': 'null'}], 'default': None, 'description': 'Extra metadata stored along with memorised information. Any json is accepted.', 'title': 'Metadata'}}, 'required': ['information'], 'type': 'object'}, outpu

In [6]:
# 에이전트에게 부여할 시스템 지시문(역할 정의) - 한글 버전
INSTRUCTIONS = """당신은 웹에서 주제를 조사하고, 나중에 활용할 지식 베이스를 구축하는 리서처입니다.
보관할 가치가 있는 내용을 알게 되면 반드시 지식 베이스에 저장하세요.
알고 있는 것이 무엇인지 질문받으면 지식 베이스를 검색해서 그 내용을 근거로 답변하세요."""

# 사용할 LLM 모델 지정
model = "gpt-5-mini"

# Tavily 웹 검색 MCP 서버를 stdio 방식으로 실행
# tool_filter=search_only : 이 서버에서 검색 관련 툴만 노출하도록 제한
# client_session_timeout_seconds=60 : 검색 응답 대기 최대 60초
async with MCPServerStdio(params=tavily_params, client_session_timeout_seconds=60, tool_filter=search_only) as search_server:
    # 벡터스토어(지식 베이스) MCP 서버를 stdio 방식으로 실행
    # 임베딩/색인 작업은 시간이 더 걸리므로 타임아웃을 120초로 설정
    async with MCPServerStdio(params=vectorstore_params, client_session_timeout_seconds=120) as vector_server:
        # 에이전트 생성 - 검색 서버와 벡터스토어 서버를 모두 도구로 연결
        agent = Agent(name="researcher", instructions=INSTRUCTIONS, model=model, mcp_servers=[search_server, vector_server])
        # trace 블록: 이 안에서 발생하는 에이전트 실행 과정을 하나의 추적 단위로 묶어 기록
        with trace("research and store"):
            # 에이전트 실행 - 한글 요청을 전달하고, 최대 20턴까지 도구 호출/추론 반복 허용
            result = await Runner.run(agent, "엔비디아(Nvidia)에 대한 최신 뉴스를 조사하고, 핵심 사실들을 지식 베이스에 저장하세요.", max_turns=20)
        # 최종 응답을 마크다운 형식으로 렌더링해 출력
        display(Markdown(result.final_output))

조사 완료했고 핵심 사실들을 지식 베이스에 저장했습니다. 저장된 항목(요약):

- 엔비디아가 허깅페이스 인수 합의(약 $12.9–13.0B, 2026년 9월 초).
- 인수에 직원 유치용 지분 기반 보상 프로그램 포함(최대 약 $1B).
- 재무(보고된 최근 분기): Q2 FY2027 매출 약 $96.2B(전년비 +106%), 데이터센터 매출 약 $89B(전년비 +117%); 다음 분기 가이던스 약 $108B ±2%; 장기 성장 가이던스(약 70%) 등(출처 표기).
- 제품/인프라: 에이전트용 CPU 'Vera' 대량 출하 시작; Vera Rubin 및 Blackwell 아키텍처 생산/램프업 진행.
- 소비자 SW: DLSS 5 및 GeForce 드라이버(616.64, 2026-09-03) 출시 — 초기 일부 타이틀에 적용.
- 전략적 의도: 허깅페이스 인수는 오픈 모델·개발자 생태계 강화 및 엔비디아의 풀스택 AI 플랫폼 확장 목적.

원하시면 각 항목의 출처 링크 원문도 첨부해 드리겠습니다.

The agent below has only the knowledge base, no web search. Whatever it tells us about Nvidia, it is recalling from what the first agent stored. That is the retrieval half of RAG.

In [7]:
# 벡터스토어(지식 베이스) MCP 서버만 stdio 방식으로 실행
# 이번에는 검색 서버 없이 저장된 지식만 조회하는 단계이므로 벡터스토어만 연결
# client_session_timeout_seconds=120 : 조회 응답 대기 최대 120초
async with MCPServerStdio(params=vectorstore_params, client_session_timeout_seconds=120) as vector_server:
    # 에이전트 생성 - 지식 베이스 서버만 도구로 연결 (웹 검색 불가)
    agent = Agent(name="researcher", instructions=INSTRUCTIONS, model=model, mcp_servers=[vector_server])
    # trace 블록: 이 안에서 발생하는 에이전트 실행 과정을 "retrieve" 라는 추적 단위로 묶어 기록
    with trace("retrieve"):
        # 에이전트 실행 - 저장된 지식 베이스를 근거로 답변하도록 한글 요청 전달
        result = await Runner.run(agent, "당신의 지식 베이스를 바탕으로, 엔비디아(Nvidia)에 대한 최신 소식은 무엇인가요?")
    # 최종 응답을 마크다운 형식으로 렌더링해 출력
    display(Markdown(result.final_output))

최근(지식베이스 기준, 2026-09-03~) 엔비디아 관련 주요 소식 요약입니다:

- Hugging Face 인수 합의: 인수가 약 129.3억 달러(≈$12.93B)로 보고되었고, 오픈 모델·개발자 생태계 통합 목적.  
- 실적 호조: FY2027 분기 실적(보고 시점 기준) 대폭 증가 — 분기 매출 약 $96.2B, 데이터센터 매출 약 $89B 등(전년대비 큰 폭 성장). 다음 분기 가이던스 약 $108B ±2% 수준 언급.  
- 제품·SW 업데이트: DLSS 5 신경 렌더링 롤아웃 및 신규 GeForce 드라이버(예: 616.64) 공개.  
- CPU·AI 인프라 확대: 에이전트형 AI 워크로드용 CPU(Vera) 및 Blackwell 아키텍처/플랫폼 생산·출하시작 언급.  
- 인수 관련 보상·유지책: Hugging Face 직원 대상 주식 기반 보상·유지 프로그램 규모 약 $10억 수준 보고.

원하시면 각 항목의 원문 출처(뉴스 링크)나 더 상세한 내용(재무 수치 출처, DLSS 지원 타이틀 등) 제공해 드리겠습니다. 어느 것을 보시겠습니까?

### Check out the trace

https://platform.openai.com/traces

## That's it for today

Four MCP servers, four kinds of context: long-term memory, web search, a knowledge base the agent builds itself, and a live integration. Choosing which sources an agent needs, and wiring them in, is much of what context engineering is in practice.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercises</h2>
            <span style="color:#ff7800;">Browse an MCP marketplace like glama.ai/mcp or smithery.ai and add another context source to this notebook, using one of today's four patterns.
            </span>
        </td>
    </tr>
</table>